# CAG (Cache-Augmented Generation) Example 01-2: Answer Generation

このノートブックでは、CAGアプローチの第二段階として、Redisキャッシュを使用して質問に対する回答を生成します。

## 処理の流れ
1. 質問の入力
2. 質問のembeddingを生成
3. キャッシュされたQ&Aペアとの類似度を計算
4. 類似度が高い場合はキャッシュから回答を取得
5. 類似度が低い場合は関連チャンクを検索してLLMで新しい回答を生成
6. 新しい回答をキャッシュに保存

## 必要なライブラリのインポート

In [ ]:
import os
import json
import hashlib
from datetime import datetime
import redis
import boto3
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd
from tqdm import tqdm
import time

## 設定の定義

In [ ]:
# AWS設定
AWS_PROFILE = os.environ['AWS_PROFILE']
REGION = 'us-east-1'
MODEL_ID = 'anthropic.claude-v2:1'
CONTENT_TYPE = 'application/json'
ACCEPT = '*/*'
RESPONSE_ITEM = 'completion'

# Redis設定
REDIS_HOST = 'llm-rag-examples-redis'
REDIS_PORT = 6379
REDIS_DB = 0

# Embedding設定
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

# 類似度の閾値
SIMILARITY_THRESHOLD = 0.75

# 検索設定
TOP_K_CHUNKS = 3

print(f"AWS Profile: {AWS_PROFILE}")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"Similarity Threshold: {SIMILARITY_THRESHOLD}")
print(f"Top K Chunks: {TOP_K_CHUNKS}")

## Redisクライアントの初期化

In [ ]:
# Redisクライアントの初期化
try:
    redis_client = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        db=REDIS_DB,
        decode_responses=True
    )
    # 接続テスト
    redis_client.ping()
    print("✓ Redis connection successful")
    use_redis = True
except Exception as e:
    print(f"✗ Redis connection failed: {e}")
    print("Using in-memory dictionary as fallback")
    redis_client = {}
    use_redis = False

print(f"Using Redis: {use_redis}")

## AWS Bedrockクライアントの初期化

In [ ]:
# AWS Bedrockクライアントの初期化
session = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)
bedrock = session.client('bedrock-runtime')
print("✓ AWS Bedrock client initialized")

## Embeddingモデルの初期化

In [ ]:
# Embeddingモデルの初期化
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL}")

## キャッシュの確認

In [ ]:
# キャッシュされたデータの確認
if use_redis:
    chunk_keys = redis_client.keys('cag_chunk:*')
    qa_keys = redis_client.keys('cag_qa:*')
else:
    all_keys = list(redis_client.keys())
    chunk_keys = [k for k in all_keys if k.startswith('cag_chunk:')]
    qa_keys = [k for k in all_keys if k.startswith('cag_qa:')]

print(f"=== Cache Status ===")
print(f"Chunk keys: {len(chunk_keys)}")
print(f"Q&A keys: {len(qa_keys)}")

if not chunk_keys and not qa_keys:
    print("\n⚠️  Warning: No cached data found!")
    print("Please run 'cag-examples01-1-register.ipynb' first to populate the cache.")
else:
    print("\n✓ Cache data found and ready to use")
    
    # キャッシュされたQ&Aの一覧を表示
    if qa_keys:
        print("\n=== Cached Q&A Pairs ===")
        for i, key in enumerate(qa_keys[:5]):
            if use_redis:
                qa_data = json.loads(redis_client.get(key))
            else:
                qa_data = json.loads(redis_client[key])
            print(f"{i+1}. {qa_data['question']}")

## 質問の定義とembedding生成

In [ ]:
# テスト用の質問を定義
test_questions = [
    "ブログの管理画面にはどうやってログインしますか？",     # キャッシュされている可能性が高い
    "記事の下書きを保存する方法を教えてください",           # 類似の質問がキャッシュされている可能性
    "ブログの投稿者の権限について教えてください",           # 新しい質問（キャッシュされていない可能性）
    "WordPressの管理画面のURLは何ですか？",              # キャッシュされている可能性が高い
    "記事を公開する際の注意点はありますか？",              # 類似の質問がキャッシュされている可能性
    "編集長の連絡先を教えてください",                    # 新しい質問
    "ブログ記事のカテゴリ分類について教えてください"       # 新しい質問
]

print(f"Test questions defined: {len(test_questions)}")
for i, q in enumerate(test_questions, 1):
    print(f"{i}. {q}")

## 質問のembeddingを生成

In [ ]:
# 質問のembeddingを生成
print("Generating embeddings for test questions...")

question_embeddings = []
for question in tqdm(test_questions, desc="Embedding questions"):
    question_embedding = embeddings.embed_query(question)
    question_embeddings.append(question_embedding)

print(f"✓ Generated {len(question_embeddings)} question embeddings")
print(f"Embedding dimension: {len(question_embeddings[0])}")

## キャッシュされたQ&Aとの類似度計算

In [ ]:
# キャッシュされたQ&Aデータを読み込み
cached_qa_data = []
print("Loading cached Q&A data...")

for key in tqdm(qa_keys, desc="Loading Q&A data"):
    if use_redis:
        qa_data = json.loads(redis_client.get(key))
    else:
        qa_data = json.loads(redis_client[key])
    cached_qa_data.append(qa_data)

print(f"✓ Loaded {len(cached_qa_data)} cached Q&A pairs")

In [ ]:
# 各質問に対して類似度を計算
print("Calculating similarity scores...")

results = []

for i, (question, question_embedding) in enumerate(zip(test_questions, question_embeddings)):
    print(f"\n{'='*50}")
    print(f"Question {i+1}: {question}")
    print(f"{'='*50}")
    
    best_similarity = 0.0
    best_match = None
    
    # キャッシュされたQ&Aとの類似度を計算
    for cached_qa in cached_qa_data:
        cached_embedding = np.array(cached_qa['question_embedding']).reshape(1, -1)
        current_embedding = np.array(question_embedding).reshape(1, -1)
        
        similarity = cosine_similarity(current_embedding, cached_embedding)[0][0]
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_match = cached_qa
    
    print(f"Best similarity: {best_similarity:.3f}")
    if best_match:
        print(f"Best match question: {best_match['question']}")
    
    result = {
        'question': question,
        'question_embedding': question_embedding,
        'best_similarity': best_similarity,
        'best_match': best_match,
        'cache_hit': best_similarity >= SIMILARITY_THRESHOLD
    }
    results.append(result)
    
    if result['cache_hit']:
        print(f"✓ Cache hit! Using cached answer.")
        print(f"Answer: {best_match['answer'][:200]}...")
    else:
        print(f"✗ Cache miss. Will need to generate new answer.")

print(f"\n✓ Similarity calculation completed for {len(results)} questions")

## キャッシュされたチャンクデータの読み込み

In [ ]:
# キャッシュされたチャンクデータを読み込み
cached_chunk_data = []
print("Loading cached chunk data...")

for key in tqdm(chunk_keys, desc="Loading chunk data"):
    if use_redis:
        chunk_data = json.loads(redis_client.get(key))
    else:
        chunk_data = json.loads(redis_client[key])
    cached_chunk_data.append(chunk_data)

print(f"✓ Loaded {len(cached_chunk_data)} cached chunks")

## 関連チャンクの検索とLLMでの回答生成

In [ ]:
# キャッシュミスした質問に対して新しい回答を生成
print("Generating new answers for cache misses...")

for i, result in enumerate(results):
    if result['cache_hit']:
        # キャッシュヒットした場合はスキップ
        result['answer'] = result['best_match']['answer']
        result['source'] = 'cache'
        result['response_time'] = 0.0
        continue
    
    print(f"\n--- Processing Question {i+1} ---")
    print(f"Question: {result['question']}")
    
    start_time = time.time()
    
    # 関連チャンクを検索
    chunk_similarities = []
    question_embedding = np.array(result['question_embedding']).reshape(1, -1)
    
    for chunk_data in cached_chunk_data:
        chunk_embedding = np.array(chunk_data['embedding']).reshape(1, -1)
        similarity = cosine_similarity(question_embedding, chunk_embedding)[0][0]
        chunk_similarities.append({
            'text': chunk_data['text'],
            'similarity': similarity,
            'chunk_id': chunk_data['chunk_id']
        })
    
    # 類似度でソートして上位を取得
    chunk_similarities.sort(key=lambda x: x['similarity'], reverse=True)
    top_chunks = chunk_similarities[:TOP_K_CHUNKS]
    
    print(f"Top {len(top_chunks)} relevant chunks found:")
    for j, chunk in enumerate(top_chunks):
        print(f"  Chunk {chunk['chunk_id']}: Similarity {chunk['similarity']:.3f}")
    
    # 関連チャンクを使ってコンテキストを作成
    context = "\n\n".join([chunk['text'] for chunk in top_chunks])
    
    # bedrock-api-ex01.ipynbを参考にしたプロンプトの構築
    prompt = f"""

Human: 以下の文書を参考に、質問に回答してください。

文書:
{context}

質問: {result['question']}

回答は具体的で実用的な内容にしてください。文書に記載されていない内容については「文書に記載されていません」と回答してください。

Assistant:"""

    # Bedrockリクエストのボディを構築
    body = {
        'prompt': prompt,
        'max_tokens_to_sample': 2048,
        'temperature': 0.3,
        'top_k': 250,
        'top_p': 1,
        'stop_sequences': ['\n\nHuman:'],
        'anthropic_version': 'bedrock-2023-05-31',
    }

    try:
        # Bedrockに回答生成をリクエスト
        response = bedrock.invoke_model(
            body=json.dumps(body),
            modelId=MODEL_ID,
            accept=ACCEPT,
            contentType=CONTENT_TYPE
        )
        
        response_body = json.loads(response.get('body').read())
        answer = response_body[RESPONSE_ITEM].strip()
        
        result['answer'] = answer
        result['source'] = 'llm'
        result['top_chunks'] = top_chunks
        
        print(f"✓ Answer generated: {answer[:200]}...")
        
    except Exception as e:
        print(f"✗ Error generating answer: {e}")
        result['answer'] = "回答の生成に失敗しました。"
        result['source'] = 'error'
        result['top_chunks'] = top_chunks
    
    result['response_time'] = time.time() - start_time
    print(f"Response time: {result['response_time']:.2f}s")
    
    # APIレート制限を避けるため少し待機
    time.sleep(1)

print("\n✓ Answer generation completed")

## 新しい回答をキャッシュに保存

In [ ]:
# 新しく生成した回答をキャッシュに保存
print("Saving new Q&A pairs to cache...")

new_qa_count = 0
for result in results:
    if result['source'] == 'llm' and result['answer'] != "回答の生成に失敗しました。":
        # キャッシュキーを生成
        hash_value = hashlib.md5(result['question'].encode('utf-8')).hexdigest()
        cache_key = f"cag_qa:{hash_value}"
        
        # キャッシュデータを構築
        cache_data = {
            'type': 'qa_pair',
            'question': result['question'],
            'answer': result['answer'],
            'question_embedding': result['question_embedding'],
            'timestamp': datetime.now().isoformat()
        }
        
        # キャッシュに保存
        if use_redis:
            redis_client.set(cache_key, json.dumps(cache_data, ensure_ascii=False))
        else:
            redis_client[cache_key] = json.dumps(cache_data, ensure_ascii=False)
        
        new_qa_count += 1
        print(f"✓ Cached new Q&A pair: {result['question'][:50]}...")

print(f"\n✓ Cached {new_qa_count} new Q&A pairs")

## 結果の表示とパフォーマンス分析

In [ ]:
# 全体の結果を表示
print("\n" + "="*80)
print("QUESTION ANSWERING RESULTS")
print("="*80)

for i, result in enumerate(results, 1):
    print(f"\n--- Question {i} ---")
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Source: {result['source']}")
    print(f"Cache Hit: {result['cache_hit']}")
    print(f"Similarity: {result['best_similarity']:.3f}")
    print(f"Response Time: {result['response_time']:.2f}s")
    if 'top_chunks' in result:
        print(f"Top Chunk Similarities: {[c['similarity'] for c in result['top_chunks'][:3]]}")

print("\n" + "="*80)

## パフォーマンス統計

In [ ]:
# パフォーマンス統計を計算
cache_hits = sum(1 for r in results if r['cache_hit'])
cache_misses = len(results) - cache_hits
total_response_time = sum(r['response_time'] for r in results)
avg_response_time = total_response_time / len(results)

cache_hit_times = [r['response_time'] for r in results if r['cache_hit']]
cache_miss_times = [r['response_time'] for r in results if not r['cache_hit']]

avg_cache_hit_time = sum(cache_hit_times) / len(cache_hit_times) if cache_hit_times else 0
avg_cache_miss_time = sum(cache_miss_times) / len(cache_miss_times) if cache_miss_times else 0

print("\n" + "="*60)
print("PERFORMANCE STATISTICS")
print("="*60)
print(f"Total Questions: {len(results)}")
print(f"Cache Hits: {cache_hits} ({cache_hits/len(results)*100:.1f}%)")
print(f"Cache Misses: {cache_misses} ({cache_misses/len(results)*100:.1f}%)")
print(f"\nResponse Times:")
print(f"  Average Overall: {avg_response_time:.2f}s")
print(f"  Average Cache Hit: {avg_cache_hit_time:.2f}s")
print(f"  Average Cache Miss: {avg_cache_miss_time:.2f}s")
if avg_cache_hit_time > 0 and avg_cache_miss_time > 0:
    speedup = avg_cache_miss_time / avg_cache_hit_time
    print(f"  Speedup with Cache: {speedup:.1f}x")
print(f"\nCache Configuration:")
print(f"  Similarity Threshold: {SIMILARITY_THRESHOLD}")
print(f"  Storage: {'Redis' if use_redis else 'In-memory dictionary'}")
print(f"  New Q&A Pairs Cached: {new_qa_count}")
print("="*60)

## 結果をCSVファイルに保存

In [ ]:
# 結果をDataFrameに変換（embeddingは除外）
results_for_csv = []
for result in results:
    row = {
        'question': result['question'],
        'answer': result['answer'],
        'source': result['source'],
        'cache_hit': result['cache_hit'],
        'best_similarity': result['best_similarity'],
        'response_time': result['response_time']
    }
    if 'best_match' in result and result['best_match']:
        row['matched_question'] = result['best_match']['question']
    else:
        row['matched_question'] = ''
    
    results_for_csv.append(row)

results_df = pd.DataFrame(results_for_csv)

print("=== Results Preview ===")
print(results_df)

# CSVファイルに保存
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
csv_filename = f'output/cag-examples01-2-results.{timestamp}.csv'
results_df.to_csv(csv_filename, index=False)
print(f"\n✓ Results saved to: {csv_filename}")

## 最新のキャッシュ統計

In [ ]:
# 最新のキャッシュ統計を表示
if use_redis:
    final_chunk_keys = redis_client.keys('cag_chunk:*')
    final_qa_keys = redis_client.keys('cag_qa:*')
else:
    final_all_keys = list(redis_client.keys())
    final_chunk_keys = [k for k in final_all_keys if k.startswith('cag_chunk:')]
    final_qa_keys = [k for k in final_all_keys if k.startswith('cag_qa:')]

print("\n=== Final Cache Statistics ===")
print(f"Total Chunk Keys: {len(final_chunk_keys)}")
print(f"Total Q&A Keys: {len(final_qa_keys)}")
print(f"New Q&A Keys Added: {len(final_qa_keys) - len(qa_keys)}")
print(f"Cache Storage: {'Redis' if use_redis else 'In-memory dictionary'}")

## 完了サマリー

In [ ]:
print("\n" + "="*80)
print("CAG ANSWER GENERATION COMPLETED")
print("="*80)
print(f"✓ Processed {len(results)} questions")
print(f"✓ Cache hit rate: {cache_hits/len(results)*100:.1f}%")
print(f"✓ Average response time: {avg_response_time:.2f}s")
print(f"✓ New Q&A pairs cached: {new_qa_count}")
print(f"✓ Results saved to: {csv_filename}")
print(f"✓ Embedding model: {EMBEDDING_MODEL}")
print(f"✓ LLM model: {MODEL_ID}")
print(f"✓ Cache storage: {'Redis' if use_redis else 'In-memory dictionary'}")
print("\nCAG system demonstrated improved response times for cached questions")
print("and automatic learning of new question-answer pairs.")
print("="*80)